# 金融数据分析与建模：作业二（1.2–1.5 数据获取）

这个 notebook 只负责 **1.2 市场指数、1.3 宏观指标、1.4 财务指标、1.5 下载日志与 README 记录**。  
你已经完成了 1.1，因此这里不重复下载股票日行情。

**运行方式：**
1. 先确保已经安装：`baostock`、`akshare`、`pandas`
2. 依次运行下面所有单元格
3. 结果会写入 `dshw-p01/` 目录下的 `data/index/`、`data/macro/`、`data/finance/`、`download_log.txt`、`README.md`

> 如果你在 1.1 使用的是我之前给你的 10 只股票，这里可以直接运行。  
> 如果你在 1.1 改过股票池，请在第二个代码单元格里同步修改 `STOCKS` 列表。


In [1]:
import sys
print(sys.version)
print(sys.executable)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
d:\Anaconda3\python.exe


In [2]:
import akshare as ak
import baostock as bs
import pandas as pd
print("依赖导入成功")

依赖导入成功


In [3]:
import sys
!{sys.executable} -m pip install akshare baostock pandas

In [4]:
from pathlib import Path
from datetime import datetime
import os
import re
import pandas as pd
import baostock as bs
import akshare as ak

START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")
FIN_YEARS = [2020, 2021, 2022, 2023, 2024]

# 如果你在 1.1 用的是我之前给你的股票池，这里不用改
# 如果你自己改过 1.1 的 10 只股票，请把下面列表同步改成一致
STOCKS = [
    {"bs_code": "sh.600036", "code": "600036", "name": "招商银行", "industry": "银行"},
    {"bs_code": "sh.601398", "code": "601398", "name": "工商银行", "industry": "银行"},
    {"bs_code": "sz.000625", "code": "000625", "name": "长安汽车", "industry": "汽车"},
    {"bs_code": "sh.600104", "code": "600104", "name": "上汽集团", "industry": "汽车"},
    {"bs_code": "sh.600519", "code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"bs_code": "sz.000858", "code": "000858", "name": "五粮液", "industry": "白酒"},
    {"bs_code": "sh.600028", "code": "600028", "name": "中国石化", "industry": "能源"},
    {"bs_code": "sh.601857", "code": "601857", "name": "中国石油", "industry": "能源"},
    {"bs_code": "sh.600050", "code": "600050", "name": "中国联通", "industry": "通讯"},
    {"bs_code": "sz.000063", "code": "000063", "name": "中兴通讯", "industry": "通讯"},
]

# 1.2 市场指数：沪深300（必选）+ 上证综指（自选）
INDICES = [
    {"bs_code": "sh.000300", "code": "000300", "name": "沪深300", "reason": "作为后续 CAPM 分析的市场基准。"},
    {"bs_code": "sh.000001", "code": "000001", "name": "上证综指", "reason": "反映沪市整体走势，可作为综合市场指数补充。"},
]

# 项目根目录：如果当前目录已经是 dshw-p01，就直接用当前目录；否则新建 dshw-p01
cwd = Path.cwd()
PROJECT_ROOT = cwd if cwd.name == "dshw-p01" else cwd / "dshw-p01"

DIRS = [
    PROJECT_ROOT / "data" / "index",
    PROJECT_ROOT / "data" / "macro",
    PROJECT_ROOT / "data" / "finance",
]

for folder in DIRS:
    os.makedirs(folder, exist_ok=True)

(PROJECT_ROOT / "download_log.txt").touch(exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("END_DATE =", END_DATE)

PROJECT_ROOT = d:\Microsoft VS Code\dshw-p01
END_DATE = 2026-04-05


In [5]:
def write_log(status: str, task_name: str, message: str) -> None:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{now}] {status:<8} {task_name:<18} {message}\n"
    with open(PROJECT_ROOT / "download_log.txt", "a", encoding="utf-8") as f:
        f.write(line)

def resultset_to_df(rs) -> pd.DataFrame:
    if rs.error_code != "0":
        raise RuntimeError(rs.error_msg)

    rows = []
    while (rs.error_code == "0") and rs.next():
        rows.append(rs.get_row_data())

    if not rows:
        return pd.DataFrame(columns=rs.fields)
    return pd.DataFrame(rows, columns=rs.fields)

def extract_first_numeric(df: pd.DataFrame, col: str):
    if df.empty or col not in df.columns:
        return pd.NA
    return pd.to_numeric(df.iloc[0][col], errors="coerce")

print("工具函数已就绪。")

工具函数已就绪。


In [6]:
def download_indices():
    """
    下载 1.2 市场指数数据：
    - 沪深300（必选）
    - 上证综指（自选）
    """
    lg = bs.login()
    if lg.error_code != "0":
        raise RuntimeError(f"baostock 登录失败：{lg.error_code}, {lg.error_msg}")

    all_frames = []

    try:
        for idx in INDICES:
            task_name = f"index_{idx['code']}"
            try:
                rs = bs.query_history_k_data_plus(
                    code=idx["bs_code"],
                    fields="date,code,open,high,low,close,volume,amount",
                    start_date=START_DATE,
                    end_date=END_DATE,
                    frequency="d",
                )
                df = resultset_to_df(rs)
                if df.empty:
                    raise ValueError("No data returned")

                df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
                df["code"] = idx["code"]
                for col in ["open", "high", "low", "close", "volume", "amount"]:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

                df = df[["date", "code", "open", "close", "high", "low", "volume", "amount"]]
                df = df.sort_values("date").reset_index(drop=True)

                save_path = PROJECT_ROOT / "data" / "index" / f"index_{idx['code']}.csv"
                df.to_csv(save_path, index=False, encoding="utf-8-sig")

                write_log("SUCCESS", task_name, f"shape={df.shape}")
                print(f"指数下载成功：{idx['name']} -> {df.shape}")
                all_frames.append(df)

            except Exception as e:
                write_log("FAILED", task_name, f"Error: {str(e)}")
                print(f"指数下载失败：{idx['name']} -> {e}")

    finally:
        bs.logout()

    return all_frames

In [7]:
def download_macro_data():
    """
    下载 1.3 宏观经济指标：
    - CPI同比增速（必选）
    - 1年期LPR（自选）
    """
    results = {}

    # CPI同比增速
    try:
        task_name = "macro_cpi_yoy"
        df_cpi = ak.macro_china_cpi_yearly().copy()

        # 尽量兼容不同版本的列名
        if "日期" in df_cpi.columns:
            date_col = "日期"
        elif "date" in df_cpi.columns:
            date_col = "date"
        else:
            date_col = df_cpi.columns[0]

        if "今值" in df_cpi.columns:
            value_col = "今值"
        elif "value" in df_cpi.columns:
            value_col = "value"
        else:
            value_col = df_cpi.columns[-1]

        df_cpi[date_col] = pd.to_datetime(df_cpi[date_col], errors="coerce")
        df_cpi[value_col] = pd.to_numeric(df_cpi[value_col], errors="coerce")
        df_cpi = df_cpi[df_cpi[date_col] >= pd.to_datetime(START_DATE)].copy()
        df_cpi = df_cpi.dropna(subset=[date_col, value_col])

        cpi_result = pd.DataFrame({
            "date": df_cpi[date_col].dt.strftime("%Y-%m-%d"),
            "indicator": "CPI同比增速",
            "value": df_cpi[value_col],
        }).sort_values("date").reset_index(drop=True)

        cpi_result.to_csv(PROJECT_ROOT / "data" / "macro" / "macro_cpi_yoy.csv",
                          index=False, encoding="utf-8-sig")
        write_log("SUCCESS", task_name, f"shape={cpi_result.shape}")
        print(f"宏观数据下载成功：CPI同比增速 -> {cpi_result.shape}")
        results["cpi"] = cpi_result

    except Exception as e:
        write_log("FAILED", "macro_cpi_yoy", f"Error: {str(e)}")
        print(f"宏观数据下载失败：CPI同比增速 -> {e}")

    # 1年期LPR
    try:
        task_name = "macro_lpr_1y"
        df_lpr = ak.macro_china_lpr().copy()

        if "TRADE_DATE" in df_lpr.columns:
            date_col = "TRADE_DATE"
        elif "date" in df_lpr.columns:
            date_col = "date"
        else:
            date_col = df_lpr.columns[0]

        if "LPR1Y" in df_lpr.columns:
            value_col = "LPR1Y"
        elif "lpr1y" in df_lpr.columns:
            value_col = "lpr1y"
        else:
            # 兜底：优先找名字里带 1Y 的列
            candidates = [c for c in df_lpr.columns if "1Y" in str(c).upper()]
            value_col = candidates[0] if candidates else df_lpr.columns[-1]

        df_lpr[date_col] = pd.to_datetime(df_lpr[date_col], errors="coerce")
        df_lpr[value_col] = pd.to_numeric(df_lpr[value_col], errors="coerce")
        df_lpr = df_lpr[df_lpr[date_col] >= pd.to_datetime(START_DATE)].copy()
        df_lpr = df_lpr.dropna(subset=[date_col, value_col])

        lpr_result = pd.DataFrame({
            "date": df_lpr[date_col].dt.strftime("%Y-%m-%d"),
            "indicator": "1年期LPR",
            "value": df_lpr[value_col],
        }).sort_values("date").reset_index(drop=True)

        lpr_result.to_csv(PROJECT_ROOT / "data" / "macro" / "macro_lpr_1y.csv",
                          index=False, encoding="utf-8-sig")
        write_log("SUCCESS", task_name, f"shape={lpr_result.shape}")
        print(f"宏观数据下载成功：1年期LPR -> {lpr_result.shape}")
        results["lpr"] = lpr_result

    except Exception as e:
        write_log("FAILED", "macro_lpr_1y", f"Error: {str(e)}")
        print(f"宏观数据下载失败：1年期LPR -> {e}")

    return results

In [8]:
def download_finance_data():
    """
    下载 1.4 财务指标：
    - ROE
    - 资产负债率
    输出为长格式：code, year, indicator, value
    """
    lg = bs.login()
    if lg.error_code != "0":
        raise RuntimeError(f"baostock 登录失败：{lg.error_code}, {lg.error_msg}")

    all_rows = []

    try:
        for stock in STOCKS:
            stock_task = f"finance_{stock['code']}"
            stock_rows = []

            try:
                for year in FIN_YEARS:
                    # ROE：利润表接口
                    rs_profit = bs.query_profit_data(code=stock["bs_code"], year=year, quarter=4)
                    profit_df = resultset_to_df(rs_profit)
                    roe_value = extract_first_numeric(profit_df, "roeAvg")
                    if pd.notna(roe_value):
                        stock_rows.append({
                            "code": stock["code"],
                            "year": year,
                            "indicator": "ROE",
                            "value": float(roe_value),
                        })

                    # 资产负债率：资产负债表接口
                    rs_balance = bs.query_balance_data(code=stock["bs_code"], year=year, quarter=4)
                    balance_df = resultset_to_df(rs_balance)
                    debt_value = extract_first_numeric(balance_df, "liabilityToAsset")
                    if pd.notna(debt_value):
                        stock_rows.append({
                            "code": stock["code"],
                            "year": year,
                            "indicator": "资产负债率",
                            "value": float(debt_value),
                        })

                if not stock_rows:
                    raise ValueError("No annual finance data returned")

                one_df = pd.DataFrame(stock_rows)[["code", "year", "indicator", "value"]]
                one_df = one_df.sort_values(["code", "year", "indicator"]).reset_index(drop=True)
                one_df.to_csv(PROJECT_ROOT / "data" / "finance" / f"finance_{stock['code']}.csv",
                              index=False, encoding="utf-8-sig")

                write_log("SUCCESS", stock_task, f"shape={one_df.shape}")
                print(f"财务数据下载成功：{stock['name']} -> {one_df.shape}")
                all_rows.append(one_df)

            except Exception as e:
                write_log("FAILED", stock_task, f"Error: {str(e)}")
                print(f"财务数据下载失败：{stock['name']} -> {e}")

    finally:
        bs.logout()

    if not all_rows:
        raise ValueError("没有成功下载到任何财务数据。")

    finance_long = pd.concat(all_rows, ignore_index=True)
    finance_long = finance_long[["code", "year", "indicator", "value"]]
    finance_long = finance_long.sort_values(["code", "year", "indicator"]).reset_index(drop=True)

    finance_long.to_csv(PROJECT_ROOT / "data" / "finance" / "finance_long.csv",
                        index=False, encoding="utf-8-sig")
    write_log("SUCCESS", "finance_long", f"shape={finance_long.shape}")
    print(f"财务数据汇总完成 -> {finance_long.shape}")

    return finance_long

In [9]:
def upsert_markdown_section(text: str, heading: str, body: str) -> str:
    """
    如果 README 中已经有该 section，则替换；否则追加。
    heading 例如：'## 1.2 市场指数'
    """
    pattern = rf"{re.escape(heading)}\n.*?(?=\n##\s|\Z)"
    replacement = heading + "\n" + body.strip() + "\n"
    if re.search(pattern, text, flags=re.S):
        return re.sub(pattern, replacement, text, flags=re.S)
    else:
        if text and not text.endswith("\n"):
            text += "\n"
        return text + "\n" + replacement

def update_readme():
    readme_path = PROJECT_ROOT / "README.md"
    if readme_path.exists():
        text = readme_path.read_text(encoding="utf-8")
    else:
        text = "# dshw-p01\n\n"

    body_12 = """
| 指数代码 | 指数名称 | 选择理由 |
|---|---|---|
| 000300 | 沪深300 | 作为后续 CAPM 分析的市场基准。 |
| 000001 | 上证综指 | 反映沪市整体走势，可作为综合市场指数补充。 |

- 数据时间范围：与股票数据一致，为 2020-01-01 至今
- 保存路径：`data/index/`
- 数据来源：`baostock`
"""

    body_13 = """
| 指标名称 | 选择理由 |
|---|---|
| CPI同比增速 | CPI 反映通胀压力，会影响实际利率、估值折现率和居民购买力，从而影响股票市场表现。 |
| 1年期LPR | LPR 会影响企业融资成本和估值贴现率，因此与股票市场定价具有直接关系。 |

- 保存路径：`data/macro/`
- 数据来源：`akshare`
"""

    body_14 = """
本部分获取 10 只股票最近 5 个完整年度（2020–2024）的两类财务指标：

- `ROE`
- `资产负债率`

并整理成长格式 `finance_long.csv`，字段为：

- `code`
- `year`
- `indicator`
- `value`

保存路径：`data/finance/`
"""

    body_15 = """
所有下载任务都会自动写入 `download_log.txt`，格式示例：

```text
[2026-04-05 10:23:45] SUCCESS  index_000300       shape=(1200, 8)
[2026-04-05 10:23:48] SUCCESS  macro_cpi_yoy      shape=(60, 3)
[2026-04-05 10:23:51] FAILED   finance_600036     Error: No data returned
```
"""

    text = upsert_markdown_section(text, "## 1.2 市场指数", body_12)
    text = upsert_markdown_section(text, "## 1.3 宏观经济指标", body_13)
    text = upsert_markdown_section(text, "## 1.4 财务指标", body_14)
    text = upsert_markdown_section(text, "## 1.5 下载日志", body_15)

    readme_path.write_text(text, encoding="utf-8")
    print(f"README 已更新：{readme_path}")

In [10]:
# 一键运行 1.2 - 1.5

index_result = download_indices()
macro_result = download_macro_data()
finance_result = download_finance_data()
update_readme()

print("\n全部完成。")
print("项目目录：", PROJECT_ROOT)
print("你现在可以检查以下文件：")
print("-", PROJECT_ROOT / "data" / "index")
print("-", PROJECT_ROOT / "data" / "macro")
print("-", PROJECT_ROOT / "data" / "finance" / "finance_long.csv")
print("-", PROJECT_ROOT / "download_log.txt")
print("-", PROJECT_ROOT / "README.md")

login success!
指数下载成功：沪深300 -> (1514, 8)
指数下载成功：上证综指 -> (1514, 8)
logout success!
宏观数据下载成功：CPI同比增速 -> (68, 3)


  0%|          | 0/4 [00:00<?, ?it/s]

宏观数据下载成功：1年期LPR -> (75, 3)
login success!
财务数据下载成功：招商银行 -> (10, 4)
财务数据下载成功：工商银行 -> (10, 4)
财务数据下载成功：长安汽车 -> (10, 4)
财务数据下载成功：上汽集团 -> (10, 4)
财务数据下载成功：贵州茅台 -> (10, 4)
财务数据下载成功：五粮液 -> (10, 4)
财务数据下载成功：中国石化 -> (10, 4)
财务数据下载成功：中国石油 -> (10, 4)
财务数据下载成功：中国联通 -> (10, 4)
财务数据下载成功：中兴通讯 -> (10, 4)
logout success!
财务数据汇总完成 -> (100, 4)
README 已更新：d:\Microsoft VS Code\dshw-p01\README.md

全部完成。
项目目录： d:\Microsoft VS Code\dshw-p01
你现在可以检查以下文件：
- d:\Microsoft VS Code\dshw-p01\data\index
- d:\Microsoft VS Code\dshw-p01\data\macro
- d:\Microsoft VS Code\dshw-p01\data\finance\finance_long.csv
- d:\Microsoft VS Code\dshw-p01\download_log.txt
- d:\Microsoft VS Code\dshw-p01\README.md
